# Compare saved SegmentEveryForam U-Net checkpoints

Run on a fresh Colab runtime. GPU is optional and speeds up evaluation. Upload a ZIP of reference image–mask pairs, such as the three pairs in `Fine Tuning/Testset2`. No training occurs. Both checkpoints use identical test patches and the repository evaluation function. This computes boundary-class IoU. Confirm the evaluation images were excluded from both actual training runs before describing them as independent test data.

The 256 × 256 patches use stride 128 and VALID padding, matching the v1 patch procedure. Overlapping pixels contribute more than once, and incomplete edge patches are omitted. The pooled result averages IoU across background, interior, and boundary classes. Loss is averaged across batches, following the existing evaluator.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.request import urlretrieve
from zipfile import ZipFile, is_zipfile
from glob import glob
import os, json, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import load_img
from PIL import Image
from tqdm.auto import tqdm
from google.colab import files

work = Path(mkdtemp(prefix="sef_evaluation_"))
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
# Upload a ZIP containing matching *_image and *_mask files.
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one ZIP containing the evaluation image–mask pairs.")
name, payload = next(iter(uploaded.items()))
bundle = work / "inputs.zip"
bundle.write_bytes(payload)
originals = work / "originals"
originals.mkdir()
with ZipFile(bundle) as archive:
    for member in archive.infolist():
        base = Path(member.filename).name
        if not member.is_dir() and Path(base).stem.lower().endswith(("_image", "_mask")):
            destination = originals / base
            if destination.exists():
                raise ValueError(f"Duplicate filename: {base}")
            destination.write_bytes(archive.read(member))
def index(suffix):
    mapping = {}
    for path in originals.iterdir():
        if path.stem.lower().endswith(suffix):
            key = path.stem[:-len(suffix)].lower()
            if key in mapping:
                raise ValueError(f"Duplicate pair: {key}")
            mapping[key] = path
    return mapping
images, masks = index("_image"), index("_mask")
if not images or images.keys() != masks.keys():
    raise ValueError("Every evaluation image must have one matching reference mask.")
for key in images:
    with Image.open(images[key]) as image, Image.open(masks[key]) as mask:
        values = np.asarray(mask.convert("RGB"))[:, :, 0]
        if image.size != mask.size or not np.isin(values, [0, 1, 2]).all():
            raise ValueError(f"Check dimensions and 0/1/2 class labels for {key}")
        if min(image.size) < 256:
            raise ValueError(f"Image is smaller than one evaluation patch: {key}")
print(f"Validated {len(images)} image–mask pairs.")


## Evaluation functions

Frozen from the repository implementation. Evaluation uses unaugmented inputs normalized to 0–1.

In [ ]:
def patchify_training_data(input_dir, patch_dir):
    """
    Extracts patches from training images and labels, and saves them to the specified directory.

    Parameters
    ----------
    input_dir : str
        The directory containing the input images and labels.
    patch_dir : str
        The directory where the patches will be saved.

    Returns
    -------
    tuple
        A tuple containing the paths to the directories where the image patches and label patches are saved.

    Notes
    -----
    - The function expects the input directory to contain files with 'image' and 'mask' in their filenames.
    - The patches are extracted with a size of 256x256 pixels and a stride of 128 pixels.
    - The function creates subdirectories 'images' and 'labels' within the specified patch directory to save the patches.
    - If the input directory does not exist or contains no matching files, the function will print a warning and return.
    """
    if not os.path.exists(input_dir):
        print(f"Warning: The directory {input_dir} does not exist.")
        return

    images = sorted(glob(input_dir + "*image*"))
    labels = sorted(glob(input_dir + "*mask*"))

    if not images or not labels:
        print(f"Warning: No files containing 'image' or 'mask' found in {input_dir}.")
        return

    # Create directories for patches
    patches_dir = os.path.join(patch_dir, "Patches")
    images_dir = os.path.join(patches_dir, "images")
    labels_dir = os.path.join(patches_dir, "labels")
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    start_no = 0
    for image in tqdm(images):
        # Load the large image
        large_image = load_img(image)
        # Convert the image to a tensor
        large_image = tf.keras.preprocessing.image.img_to_array(large_image)
        # Reshape the tensor to have a batch size of 1
        large_image = tf.reshape(large_image, [1, *large_image.shape])
        # Extract patches from the large image
        patches = tf.image.extract_patches(
            images=large_image,
            sizes=[1, 256, 256, 1],
            strides=[1, 128, 128, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        # Reshape the patches tensor to have a batch size of -1
        patches = tf.reshape(patches, [-1, 256, 256, 3])
        # Write patches to files
        for i in range(patches.shape[0]):
            im = np.asarray(patches[i, :, :, :]).astype("uint8")
            imname = os.path.join(images_dir, "im%03d.png" % (start_no + i))
            im = Image.fromarray(im.astype(np.uint8))
            im.save(imname)
        start_no = start_no + patches.shape[0]

    start_no = 0
    for image in tqdm(labels):
        # Load the large image
        large_image = load_img(image)
        # Convert the image to a tensor
        large_image = tf.keras.preprocessing.image.img_to_array(large_image)
        large_image = large_image[
            :, :, 0, np.newaxis
        ]  # only keep one layer and add a new axis
        # Reshape the tensor to have a batch size of 1
        large_image = tf.reshape(large_image, [1, *large_image.shape])
        # Extract patches from the large image
        patches = tf.image.extract_patches(
            images=large_image,
            sizes=[1, 256, 256, 1],
            strides=[1, 128, 128, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        # Reshape the patches tensor to have a batch size of -1
        patches = tf.reshape(patches, [-1, 256, 256, 1])
        # Write patches to files
        for i in range(patches.shape[0]):
            im = np.asarray(patches[i, :, :, 0]).astype("uint8")
            imname = os.path.join(labels_dir, "im%03d.png" % (start_no + i))
            im = Image.fromarray(im.astype(np.uint8))
            im.save(imname)
        start_no = start_no + patches.shape[0]
    return images_dir, labels_dir

def load_and_preprocess(image_path, mask_path, augmentations=False):
    """
    Load and preprocess an image and its corresponding mask.

    Parameters
    ----------
    image_path : str
        The file path of the image.
    mask_path : str
        The file path of the mask.
    augmentations : bool, optional
        Whether to apply augmentations to the image. Defaults to False.

    Returns
    -------
    image : numpy.ndarray
        the preprocessed image.
    mask : numpy.ndarray
        the preprocessed mask.
    """
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)

    # Load mask and one-hot encode it
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.cast(mask, tf.int32)
    mask = tf.one_hot(mask, depth=3, axis=-1)
    mask = tf.reshape(mask, (256, 256, 3))

    # Normalize images
    image = tf.cast(image, tf.float32) / 255.0

    # Apply augmentations
    if augmentations:
        # --- Geometric augmentations (applied to both image and mask) ---
        # Concatenate along channels so identical transforms apply to both
        combined = tf.concat([image, mask], axis=-1)  # shape: (256, 256, 6)

        # Random horizontal flip
        seed_hflip = tf.random.uniform(shape=[], minval=0, maxval=1)
        combined = tf.cond(
            seed_hflip < 0.5,
            lambda: tf.image.flip_left_right(combined),
            lambda: combined,
        )

        # Random vertical flip
        seed_vflip = tf.random.uniform(shape=[], minval=0, maxval=1)
        combined = tf.cond(
            seed_vflip < 0.5,
            lambda: tf.image.flip_up_down(combined),
            lambda: combined,
        )

        # Random 90-degree rotations (0, 1, 2, or 3 times)
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
        combined = tf.image.rot90(combined, k=k)

        # Random scaling/zoom (crop and resize)
        seed_scale = tf.random.uniform(shape=[], minval=0, maxval=1)
        def apply_random_scale(combined):
            scale = tf.random.uniform(shape=[], minval=0.7, maxval=0.95)
            crop_size = tf.cast(tf.round(256.0 * scale), tf.int32)
            max_offset = tf.maximum(256 - crop_size, 1)
            offset_h = tf.random.uniform(shape=[], minval=0,
                                         maxval=max_offset, dtype=tf.int32)
            offset_w = tf.random.uniform(shape=[], minval=0,
                                         maxval=max_offset, dtype=tf.int32)
            combined_cropped = tf.image.crop_to_bounding_box(
                combined, offset_h, offset_w, crop_size, crop_size
            )
            return tf.image.resize(combined_cropped, [256, 256],
                                   method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
        combined = tf.cond(
            seed_scale < 0.5, lambda: apply_random_scale(combined), lambda: combined
        )

        # Split back into image and mask
        image = combined[:, :, :3]
        mask = combined[:, :, 3:]

        # Re-discretize mask after geometric transforms (nearest-neighbor keeps it
        # clean, but this guards against any floating-point drift)
        mask = tf.round(mask)

        # --- Color augmentations (applied to image only) ---
        # Each augmentation applied independently with 50% probability

        # Random brightness
        seed_bright = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_bright < 0.5,
            lambda: tf.image.random_brightness(image, max_delta=0.2),
            lambda: image,
        )

        # Random contrast
        seed_contrast = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_contrast < 0.5,
            lambda: tf.image.random_contrast(image, lower=0.8, upper=1.2),
            lambda: image,
        )

        # Random saturation
        seed_sat = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_sat < 0.5,
            lambda: tf.image.random_saturation(image, lower=0.7, upper=1.3),
            lambda: image,
        )

        # Random hue
        seed_hue = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_hue < 0.5,
            lambda: tf.image.random_hue(image, max_delta=0.05),
            lambda: image,
        )

        # Gaussian noise
        seed_noise = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_noise < 0.5,
            lambda: image + tf.random.normal(tf.shape(image), mean=0.0, stddev=0.02),
            lambda: image,
        )

        # Gaussian blur
        seed_blur = tf.random.uniform(shape=[], minval=0, maxval=1)
        def apply_gaussian_blur(img):
            sigma = tf.random.uniform(shape=[], minval=0.5, maxval=1.5)
            kernel_size = 5
            ax = tf.cast(tf.range(-kernel_size // 2 + 1, kernel_size // 2 + 1),
                         tf.float32)
            xx, yy = tf.meshgrid(ax, ax)
            kernel = tf.exp(-(xx ** 2 + yy ** 2) / (2.0 * sigma ** 2))
            kernel = kernel / tf.reduce_sum(kernel)
            kernel = tf.reshape(kernel, [kernel_size, kernel_size, 1, 1])
            kernel = tf.tile(kernel, [1, 1, 3, 1])
            img = tf.expand_dims(img, axis=0)
            img = tf.nn.depthwise_conv2d(img, kernel, strides=[1, 1, 1, 1],
                                         padding='SAME')
            return tf.squeeze(img, axis=0)
        image = tf.cond(
            seed_blur < 0.3, lambda: apply_gaussian_blur(image), lambda: image
        )

        # Clip to valid range
        image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask

def weighted_crossentropy(y_true, y_pred, label_smoothing: float = 0.1):
    """
    Calculates the weighted cross-entropy loss between the true labels and predicted labels.

    Uses label smoothing to prevent overconfident predictions and improve
    generalization. Expects logits (pre-softmax) as y_pred.

    Parameters
    ----------
    y_true : tensor
        True labels (one-hot encoded).
    y_pred : tensor
        Predicted logits (pre-softmax).
    label_smoothing : float
        Label smoothing factor. 0.0 means no smoothing.

    Returns
    -------
    loss : tensor
        Weighted cross-entropy loss.
    """
    num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
    y_true_smooth = y_true * (1.0 - label_smoothing) + label_smoothing / num_classes
    class_weights = tf.constant(
        [[[[0.6, 1.0, 5.0]]]]
    )  # increase the weight on the grains and the grain boundaries
    unweighted_losses = tf.nn.softmax_cross_entropy_with_logits(
        labels=y_true_smooth, logits=y_pred
    )
    weights = tf.reduce_sum(class_weights * y_true, axis=-1)
    weighted_losses = weights * unweighted_losses
    loss = tf.reduce_mean(weighted_losses)
    return loss

def evaluate_model(model, dataset, class_names: list[str] | None = None):
    """
    Evaluate a trained U-Net model on a dataset, reporting overall accuracy,
    loss, and per-class IoU scores.

    Parameters
    ----------
    model : tf.keras.Model
        The trained model.
    dataset : tf.data.Dataset
        The dataset to evaluate on (e.g. the test dataset).
    class_names : list of str, optional
        Names for each output class. Defaults to
        ``["background", "grain", "boundary"]``.

    Returns
    -------
    results : dict
        Dictionary with keys ``"loss"``, ``"accuracy"``, ``"mean_iou"``,
        and ``"per_class_iou"`` (a dict mapping class name to IoU).
    """
    if class_names is None:
        class_names = ["background", "grain", "boundary"]
    num_classes = len(class_names)
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    n_batches = 0
    for images, masks in tqdm(dataset, desc="evaluating model"):
        preds = model.predict(images, verbose=0)
        # accumulate loss
        total_loss += weighted_crossentropy(masks, preds).numpy()
        n_batches += 1
        # convert to class indices
        pred_classes = np.argmax(preds, axis=-1).ravel()
        true_classes = np.argmax(masks.numpy(), axis=-1).ravel()
        total_correct += np.sum(pred_classes == true_classes)
        total_pixels += len(pred_classes)
        # accumulate confusion matrix
        for true_c in range(num_classes):
            for pred_c in range(num_classes):
                confusion[true_c, pred_c] += np.sum(
                    (true_classes == true_c) & (pred_classes == pred_c)
                )
    # compute per-class IoU from confusion matrix
    per_class_iou = {}
    for c in range(num_classes):
        tp = confusion[c, c]
        fn = np.sum(confusion[c, :]) - tp
        fp = np.sum(confusion[:, c]) - tp
        iou = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else 0.0
        per_class_iou[class_names[c]] = iou

    accuracy = total_correct / total_pixels if total_pixels > 0 else 0.0
    mean_iou = np.mean(list(per_class_iou.values()))
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0

    print(f"\nTest loss: {avg_loss:.4f}")
    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")
    for name, iou in per_class_iou.items():
        print(f"  {name} IoU: {iou:.4f}")

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "mean_iou": mean_iou,
        "per_class_iou": per_class_iou,
    }

In [ ]:
datasets = {}
for number, key in enumerate(sorted(images)):
    directory = work / f"pair_{number}"
    directory.mkdir()
    for label, mapping in [("image", images), ("mask", masks)]:
        path = mapping[key]
        shutil.copyfile(path, directory / f"specimen_{label}{path.suffix}")
    image_dir, mask_dir = patchify_training_data(str(directory) + os.sep, str(directory / "patches"))
    image_paths = sorted(glob(os.path.join(image_dir, "*.png")))
    mask_paths = sorted(glob(os.path.join(mask_dir, "*.png")))
    if not image_paths or len(image_paths) != len(mask_paths):
        raise ValueError(f"Missing patches for {key}")
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    ds = ds.map(lambda x, y: load_and_preprocess(x, y, False))
    datasets[key] = ds.batch(32).prefetch(tf.data.AUTOTUNE)


In [ ]:
checkpoint_commit = "1f311172938d2265de0b2d54f05d1244f619057b"
base = f"https://raw.githubusercontent.com/Geogabriel/SegmentEveryForam/{checkpoint_commit}/models/"
rows, detailed = [], {}
for filename in ["seg_model_foram_v1_30epochs.keras", "best_val_loss.keras"]:
    path = work / filename
    urlretrieve(base + filename, path)
    if not is_zipfile(path):
        raise ValueError(f"Invalid Keras download: {filename}")
    model = tf.keras.models.load_model(path, compile=False)
    detailed[filename] = {}
    combined = None
    for image_id, dataset in datasets.items():
        print(filename, image_id)
        result = evaluate_model(model, dataset)
        detailed[filename][image_id] = result
        rows.append(dict(model=filename, image=image_id, mean_iou=result["mean_iou"],
                         **result["per_class_iou"], accuracy=result["accuracy"], loss=result["loss"]))
        combined = dataset if combined is None else combined.concatenate(dataset)
    result = evaluate_model(model, combined)
    detailed[filename]["ALL_IMAGES"] = result
    rows.append(dict(model=filename, image="ALL_IMAGES", mean_iou=result["mean_iou"],
                     **result["per_class_iou"], accuracy=result["accuracy"], loss=result["loss"]))
    del model
    tf.keras.backend.clear_session()
table = pd.DataFrame(rows)
display(table)
output = work / "results"
output.mkdir()
table.to_csv(output / "unet_iou_comparison.csv", index=False)
(output / "metrics.json").write_text(json.dumps(detailed, indent=2))
(output / "protocol.json").write_text(json.dumps({
    "checkpoint_commit": checkpoint_commit,
    "images": [images[k].name for k in sorted(images)],
    "patch_size": 256, "stride": 128, "padding": "VALID",
    "prediction": "argmax across 3 U-Net classes; no augmentation",
    "overall": "confusion matrix pooled across all evaluation patches",
    "boundary": "IoU of the boundary class",
    "scope": "U-Net alone"
}, indent=2))
archive = shutil.make_archive(str(work / "unet_evaluation_results"), "zip", output)
files.download(archive)
